In [1]:
from loqs.backends import QSimQuantumState, PyGSTiNoiseModel
from loqs.core import QuantumProgram

from loqs.codepacks import d3_5q_code

import pygsti

In [2]:
code_5q = d3_5q_code.create_qec_code()

In [3]:
code_5q.instructions.keys()

dict_keys(['Non-FT Minus Prep', 'FT Minus Prep', 'X', 'Z', 'H', 'Logical Prime Basis Transform', 'Logical Prime Basis Inverse Transform', 'Non-FT Logical Z Measure', 'Non-FT Logical X Measure', 'Non-FT Minus Unprep', 'Adaptive Measure Termination', 'Adaptive Measure Part III', 'Adaptive Measure Part II', 'Adaptive Measure Part I', 'Adaptive Measure'])

In [4]:
# Define a pyGSTi processor spec and noise model
# This is the only pyGSTi required here,
# and could eventually be traded out for something else
qubits = ["A0", "A1"] + [f"D{i+2}" for i in range(5)]
gate_names = ["Gxpi", "Gypi", "Gzpi", "Gzpi2", "Gzmpi2", "Gh", "Gcnot", "Gcphase"]

# TODO: Currently Iz does not need to be set here
# This is because QSimQuantumState actually does not try to pull the rep
# Otherwise, this would result in a KeyError in PyGSTiNoiseModel.get_reps(),
# since it technically should be provided
pspec = pygsti.processors.QubitProcessorSpec(
    len(qubits), 
    gate_names=gate_names,
    qubit_labels=qubits,
    availability={k: "all-permutations" for k in gate_names}
)

ideal_model_pygsti = pygsti.models.create_crosstalk_free_model(pspec)

ideal_model = PyGSTiNoiseModel(ideal_model_pygsti)


In [5]:
# Logical quantum program information
patch_types = {"5Q": code_5q}

# Stack
# The "Init *" instructions will be generated by the QuantumProgram based on the values
# of state_type and patch_types below
stack = [
    ("Init State", None, (len(qubits),), {"qubit_labels": qubits}), # Autogenerated from state_type
    ("Init Patch 5Q", None, ("L0", qubits)), # Autogenerated from patch_types. Note that 5Q here must be a key
    ("Non-FT Minus Prep", "L0"),
    ("H", "L0"),
    ("X", "L0"),
    ("Non-FT Logical Z Measure", "L0")
]

program = QuantumProgram(
    stack,
    default_noise_model=ideal_model,
    state_type=QSimQuantumState,
    patch_types=patch_types,
    name="Test program"
)

In [6]:
# And now we can run it for real!
program.run(shots=100)

/Users/sserita/Documents/repos/loqs-public/loqs/core/instructions/instruction.py:94: UserWarning: Skipping param priority for kwargs
  warnings.warn(f"Skipping param priority for {key}")


In [7]:
from collections import Counter

# collect_shot_data lets us pull out keys from the histories of every shot
# In this case, we pull logical_measurement from the last frame (-1) of each shot
# We pass it into Counter to get more convenient 0/1 counts
Counter(program.collect_shot_data("logical_measurement", -1))

Counter({0: 100})

In [8]:
# Make a version that stays in the minus state
stack = [
    ("Init State", None, (len(qubits),), {"qubit_labels": qubits}), # Autogenerated from state_type
    ("Init Patch 5Q", None, ("L0", qubits)), # Autogenerated from patch_types. Note that 5Q here must be a key
    ("Non-FT Minus Prep", "L0"),
    ("Non-FT Logical X Measure", "L0")
]

program2 = QuantumProgram.from_quantum_program(program, stack)

program2.run(shots=100)

Counter(program2.collect_shot_data("logical_measurement", -1))

/Users/sserita/Documents/repos/loqs-public/loqs/core/instructions/instruction.py:94: UserWarning: Skipping param priority for kwargs
  warnings.warn(f"Skipping param priority for {key}")


Counter({1: 100})

In [10]:
# Now let's stay in minus but measure in Z
stack = [
    ("Init State", None, (len(qubits),), {"qubit_labels": qubits}), # Autogenerated from state_type
    ("Init Patch 5Q", None, ("L0", qubits)), # Autogenerated from patch_types. Note that 5Q here must be a key
    ("Non-FT Minus Prep", "L0"),
    ("Non-FT Logical Z Measure", "L0")
]

program3 = QuantumProgram.from_quantum_program(program, stack)

program3.run(shots=1000) # Let's also collect more statistics!

Counter(program3.collect_shot_data("logical_measurement", -1))

Counter({0: 531, 1: 469})